[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/basic_classification.ipynb)

# GP classification with `gpc` (EP / Laplace)

GPc port of [`GPy/basic_classification.ipynb`](../GPy/basic_classification.ipynb) (James Hensman / Alan Saul).

Gaussian-process classification is the classic case where the likelihood is **not conjugate** to the GP prior: labels are Bernoulli, so we need an approximate posterior over the latent function. GPy’s tutorial starts with expectation propagation (EP); we do the same, then show the Laplace approximation on the same data ([CIP-0036](https://github.com/SheffieldML/GPc/blob/main/cip/cip0036.md); EP path [CIP-0033](https://github.com/SheffieldML/GPc/blob/main/cip/cip0033.md)).

```python
m = gpc.GPClassification(X, Y, kernel="rbf", inference="ep")  # or "laplace"
m.optimize()
p = m.predict_proba(Xtest)
mu, var = m.predict_noiseless(Xtest)  # latent f_*
```

**Notes vs GPy**

* Requires `gpc>=0.1.0a10` (Laplace predictive + naming aliases).
* FTC (dense) only — sparse EP is out of v1. For large \(N\), use the native **IVM** tutorials.
* Labels in `{0,1}` or `{-1,1}`.
* `objective()` is **−log Z** (approx.) for SCG minimisation (GPy often prints a log marginal).
* No auto-White: softness lives in EP sites / the Laplace Hessian ([cgp_ep_classification](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/cgp_ep_classification.md)).
* No `m.plot` / `m.plot_f`: we plot `predict_proba` and latent bands with matplotlib.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")



import numpy as np

import gpc

from matplotlib import pyplot as plt



try:

    from scipy.stats import norm

except ImportError:

    import subprocess, sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])

    from scipy.stats import norm


## Generative model

Classification treats labels as draws from a conditional given \(\mathbf{X}\). In the GP model a **latent** function \(f\sim\mathcal{GP}\) takes continuous values; the probit \(\Phi(f)\) squashes them into \([0,1]\); Bernoulli draws then yield labels in \(\{0,1\}\).

Below we sample that generative process: draw \(f\) from an RBF GP, squash with \(\Phi\), then sample Bernoulli labels — the same story as the GPy twin.


In [ ]:
rng = np.random.default_rng(1)
N = 80
X = rng.uniform(0.0, 1.0, size=(N, 1))

# Latent GP draw via Cholesky of an RBF Gram (lengthscale / variance as below).
lengthscale, variance = 0.2, 7.0
d2 = (X - X.T) ** 2
K = variance * np.exp(-0.5 * d2 / lengthscale**2) + 1e-6 * np.eye(N)
f = rng.multivariate_normal(np.zeros(N), K)

p_true = norm.cdf(f)  # probit link
Y = (rng.random(N) < p_true).astype(float)[:, None]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True)
axes[0].plot(X[:, 0], f, "bo", ms=4)
axes[0].set_title("latent $f(x)$")
axes[1].plot(X[:, 0], p_true, "ro", ms=4)
axes[1].set_title(r"probit $\Phi(f)$")
axes[2].plot(X[:, 0], Y[:, 0], "kx", ms=5)
axes[2].set_ylim(-0.1, 1.1)
axes[2].set_title("Bernoulli labels")
for ax in axes:
    ax.set_xlabel("$x$")
plt.tight_layout()
plt.show()


## Inference with expectation propagation

Given the binary labels, can we recover the latent function, the class probabilities, and sensible kernel hyperparameters?

EP fits a Gaussian approximation to \(p(f\mid y)\). While SCG updates kernel parameters, the EP site parameters are held fixed; after each hyperparameter step the sites are refreshed. That interleaving is the same idea as in GPy: optimise the approximate marginal \(p(y\mid\mathcal{M})\) (here exposed as `objective()` = **−log Z**).

We start from deliberately imperfect lengthscale / variance so the pre-optimise fit is visibly wrong.


In [ ]:
k = gpc.kern.RBF(1, variance=1.0, lengthscale=0.2)
m_ep = gpc.GPClassification(X, Y, kernel=k, inference="ep")
print(m_ep)
print("paths:", m_ep.param_paths())
print("objective before:", m_ep.objective())
m_ep.optimize(iters=80, messages=True)
print("objective after:", m_ep.objective())
print(m_ep)


Predictive class probabilities on a grid (probit of the EP latent mean / variance):

In [ ]:
Xtest = np.linspace(0.0, 1.0, 200)[:, None]
p_ep = m_ep.predict_proba(Xtest)
mu_f, var_f = m_ep.predict_noiseless(Xtest)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(X[:, 0], Y[:, 0], "kx", label="data", zorder=3)
ax.plot(Xtest[:, 0], p_ep[:, 0], "b-", label=r"$p(y=1\mid x)$ EP")
ax.fill_between(
    Xtest[:, 0],
    norm.cdf((mu_f[:, 0] - 2 * np.sqrt(var_f[:, 0]))),
    norm.cdf((mu_f[:, 0] + 2 * np.sqrt(var_f[:, 0]))),
    color="C0",
    alpha=0.2,
    label=r"$\Phi(\mu_f \pm 2\sigma_f)$",
)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("$x$")
ax.set_ylabel("probability")
ax.legend(loc="best")
ax.set_title("EP classification")
plt.show()


## Laplace approximation

GPy’s classification guide notes Laplace (and variational methods) as the other classical routes to \(p(f\mid y)\). Here we fit the **same** data with `inference="laplace"`: a mode-based Gaussian approximation that is often close to EP on easy 1-D problems and cheaper per iteration. Compare predictive probabilities — not the objectives directly across methods without care, since both are approximate.


In [ ]:
m_lp = gpc.GPClassification(X, Y, kernel=gpc.kern.RBF(1, variance=1.0, lengthscale=0.2), inference="laplace")
m_lp.optimize(iters=80, messages=True)
p_lp = m_lp.predict_proba(Xtest)
print("EP objective:", m_ep.objective(), "Laplace objective:", m_lp.objective())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(X[:, 0], Y[:, 0], "kx", label="data", zorder=3)
ax.plot(Xtest[:, 0], p_ep[:, 0], "b-", label="EP")
ax.plot(Xtest[:, 0], p_lp[:, 0], "r--", label="Laplace")
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="best")
ax.set_title("EP vs Laplace predictive probabilities")
ax.set_xlabel("$x$")
plt.show()


## When to use which path

| Path | Use when |
|---|---|
| `GPClassification` + EP | Dense \(N\); GPy default parity |
| `GPClassification` + Laplace | Dense \(N\); cheaper mode-based approx |
| Native IVM + probit | Large \(N\) / active set — see GPc [ivm_unitsquare](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/ivm_unitsquare.md) |

Sparse EP inside `CGp` is not in v1.
